# Planck Spectroscopy Routine for JPA and LKIPA Noise Calibration
---

We ramp up the MXC Heater power until the temperature is stabilized, then keep the heater on at the same power and run the planck spec scripts of the JPA and the LKIPA

- $\sim 40$ data points from $10 mK - 200 mK$
- $\sim 20$ data points from $200 mK - 700 mK$

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time
import os
import h5py
import inspect
from tqdm import tqdm
import sys
import math
import presto
from presto import lockin, utils, hardware
from presto.hardware import AdcFSample, AdcMode, DacFSample, DacMode
from blueftc.BlueforsController import BlueFTController
# Reload credentials module to get latest changes
import importlib
import power_ramp as pr
importlib.reload(pr)

# Timeout for hardware communication
import requests

# BASE URL for interacting with BlueFTC
base_url = 'http://192.168.88.25:5001'

# Print JSON 
def print_JSON(r):
    r_dict = r.json()
    for key, value in r_dict.items():
        print(f"{key}: {value}")

## 1. MXC Heater Power List

In [2]:
MXC_Heater_Power_list_low_temp = 10 * np.arange(    # from 10mk - 100mk, in steps of 10 mu W
    start= 1,
    stop = 31,
    step = 1,
) 

MXC_Heater_Power_list_medium_temp = MXC_Heater_Power_list_low_temp[-1] + 50 * np.arange(    # from 100mk - 200mk, in steps of 50 mu W
    start= 1,
    stop = 15,
    step = 1,
) 

MXC_Heater_Power_list_high_temp = MXC_Heater_Power_list_medium_temp[-1] + 200 * np.arange(    # from 200mk - 700mk, in steps of 50 mu W
    start= 1,
    stop = 17,
    step = 1,
) 

MXC_Heater_Power_list = 1e-6 * np.concatenate(( # mu W
    MXC_Heater_Power_list_low_temp, 
    MXC_Heater_Power_list_medium_temp, 
    MXC_Heater_Power_list_high_temp
))

print((MXC_Heater_Power_list))

[1.0e-05 2.0e-05 3.0e-05 4.0e-05 5.0e-05 6.0e-05 7.0e-05 8.0e-05 9.0e-05
 1.0e-04 1.1e-04 1.2e-04 1.3e-04 1.4e-04 1.5e-04 1.6e-04 1.7e-04 1.8e-04
 1.9e-04 2.0e-04 2.1e-04 2.2e-04 2.3e-04 2.4e-04 2.5e-04 2.6e-04 2.7e-04
 2.8e-04 2.9e-04 3.0e-04 3.5e-04 4.0e-04 4.5e-04 5.0e-04 5.5e-04 6.0e-04
 6.5e-04 7.0e-04 7.5e-04 8.0e-04 8.5e-04 9.0e-04 9.5e-04 1.0e-03 1.2e-03
 1.4e-03 1.6e-03 1.8e-03 2.0e-03 2.2e-03 2.4e-03 2.6e-03 2.8e-03 3.0e-03
 3.2e-03 3.4e-03 3.6e-03 3.8e-03 4.0e-03 4.2e-03]


## 2. Ramp up power and capture measurements for each temperature

In [3]:
pr.heater_ramp_up(
    MXC_heater_power_list=MXC_Heater_Power_list,
    N_runs=50
)

Experiment over, turning OFF heater... 

Requesting... http://192.168.88.25:5001/heater/update 

Heater successfully turned off. 

